### IAA Analaysis

In [56]:
from bs4 import BeautifulSoup
from collections import defaultdict
from typing import Dict, List, Tuple
import os


class Span:
    """Represents an annotated span with context for unique identification."""
    
    def __init__(self, text: str, start: int, end: int, labelname: str, 
                 attributes: Dict[str, str], context_text: str = ""):
        self.text = text.strip()
        self.start = start
        self.end = end
        self.labelname = labelname
        self.attributes = attributes
        self.context_text = context_text  # Surrounding text for unique location identification
        
    def __repr__(self):
        return f"Span('{self.text[:30]}...', label={self.labelname})"
    
    def context_match(self, other: 'Span') -> bool:
        """
        Check if two spans match based on their surrounding context.
        This ensures we're comparing the SAME location in the document.
        """
        if self.labelname != other.labelname:
            return False
        
        if not self.context_text or not other.context_text:
            return False
        
        # Normalize and compare contexts
        context1 = ' '.join(self.context_text.split()).lower()
        context2 = ' '.join(other.context_text.split()).lower()
        return context1 == context2
    
    def context_overlap(self, other: 'Span', threshold: float = 0.7) -> bool:
        """
        Check if two spans have similar surrounding context (allows partial matches).
        Useful for cases where annotators might have slightly different boundaries.
        """
        if self.labelname != other.labelname:
            return False
        
        if not self.context_text or not other.context_text:
            return False
        
        # Get word sets for both contexts
        words1 = set(self.context_text.lower().split())
        words2 = set(other.context_text.lower().split())
        
        if not words1 or not words2:
            return False
        
        # Calculate Jaccard similarity
        intersection = len(words1 & words2)
        union = len(words1 | words2)
        
        if union == 0:
            return False
        
        similarity = intersection / union
        return similarity >= threshold


def get_context_around_element(element, plain_text: str, context_chars: int = 200) -> Tuple[str, int, int]:
    """
    Extract context text around an element to uniquely identify its location.
    
    The context includes text BEFORE and AFTER the labeled span, making it
    long enough to be unique in the document even if the label text itself repeats.
    
    Args:
        element: BeautifulSoup element (manual_label)
        plain_text: Plain text of the entire document
        context_chars: Number of characters to include before and after (default: 200)
        
    Returns:
        Tuple of (context_text, start_position, end_position)
    """
    # Get the element's text (normalized)
    elem_text = element.get_text()
    normalized_elem_text = ' '.join(elem_text.split())
    
    if not normalized_elem_text:
        return "", 0, 0
    
    # Normalize the plain text
    normalized_plain = ' '.join(plain_text.split())
    
    # Find the element text in the plain text
    start_idx = normalized_plain.find(normalized_elem_text)
    
    if start_idx == -1:
        # Try with first 50 characters if full text not found
        search_text = normalized_elem_text[:50] if len(normalized_elem_text) > 50 else normalized_elem_text
        start_idx = normalized_plain.find(search_text)
        if start_idx == -1:
            # Fallback: use the text itself as context
            return normalized_elem_text, 0, len(normalized_elem_text)
    
    end_idx = start_idx + len(normalized_elem_text)
    
    # Extract context: text before + element text + text after
    # This makes the context unique even if the label text appears multiple times
    context_start = max(0, start_idx - context_chars)
    context_end = min(len(normalized_plain), end_idx + context_chars)
    
    context_text = normalized_plain[context_start:context_end]
    
    return context_text, start_idx, end_idx


def extract_spans_from_html(html_file: str, context_chars: int = 200) -> List[Span]:
    """
    Extract all manual_label spans from an HTML file with surrounding context.
    
    Process:
    1. Extract plain text from the document body (no HTML tags)
    2. For each manual_label, capture text before + label text + text after
    3. Create spans with this context for accurate cross-document matching
    
    Args:
        html_file: Path to the HTML file
        context_chars: Number of characters before/after for context (default: 200)
        
    Returns:
        List of Span objects with context information
    """
    with open(html_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    soup = BeautifulSoup(content, 'html.parser')
    body = soup.find('body')
    
    if not body:
        return []
    
    # Get plain text from body (all HTML stripped)
    plain_text = body.get_text()
    
    # Find all manual_label elements
    labels = body.find_all('manual_label')
    
    spans = []
    
    for label in labels:
        labelname = label.get('labelname', '')
        
        # Get all attributes except style, labelname
        attributes = {k: v for k, v in label.attrs.items() 
                     if k not in ['style', 'labelname']}
        
        # Get the label's text content (normalized)
        text = label.get_text()
        normalized_text = ' '.join(text.split())
        
        if not normalized_text:
            continue
        
        # Extract context around this label - THIS IS THE KEY INNOVATION
        context_text, start_idx, end_idx = get_context_around_element(label, plain_text, context_chars)
        
        span = Span(normalized_text, start_idx, end_idx, labelname, attributes, context_text)
        spans.append(span)
    
    return spans


def calculate_iaa_metrics(spans1: List[Span], spans2: List[Span], 
                          match_type: str = 'context') -> Dict:
    """
    Calculate IAA metrics using context-aware matching.
    
    Args:
        spans1: Spans from annotator 1 (with context)
        spans2: Spans from annotator 2 (with context)
        match_type: 'context' for exact context match (strict),
                   'context_overlap' for similar context match (lenient)
        
    Returns:
        Dictionary with metrics: precision, recall, f1, matched counts
    """
    if match_type == 'context':
        # Exact context match: same surrounding text and label
        matched = 0
        matched_indices = set()
        
        for s1 in spans1:
            for j, s2 in enumerate(spans2):
                if j not in matched_indices and s1.context_match(s2):
                    matched += 1
                    matched_indices.add(j)
                    break
        
        precision = matched / len(spans1) if spans1 else 0
        recall = matched / len(spans2) if spans2 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        return {
            'matched': matched,
            'annotator1_count': len(spans1),
            'annotator2_count': len(spans2),
            'precision': precision,
            'recall': recall,
            'f1': f1
        }
    
    elif match_type == 'context_overlap':
        # Context-based overlap: similar surrounding text
        matched1 = set()
        matched2 = set()
        
        for i, s1 in enumerate(spans1):
            for j, s2 in enumerate(spans2):
                if s1.context_overlap(s2):
                    matched1.add(i)
                    matched2.add(j)
        
        precision = len(matched1) / len(spans1) if spans1 else 0
        recall = len(matched2) / len(spans2) if spans2 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        return {
            'matched_annotator1': len(matched1),
            'matched_annotator2': len(matched2),
            'annotator1_count': len(spans1),
            'annotator2_count': len(spans2),
            'precision': precision,
            'recall': recall,
            'f1': f1
        }
    
    else:
        raise ValueError(f"Unknown match_type: {match_type}. Use 'context' or 'context_overlap'")


def calculate_per_label_iaa(spans1: List[Span], spans2: List[Span], 
                            match_type: str = 'context') -> Dict[str, Dict]:
    """
    Calculate IAA metrics per label type.
    
    Args:
        spans1: Spans from annotator 1
        spans2: Spans from annotator 2
        match_type: 'context' or 'context_overlap'
        
    Returns:
        Dictionary mapping label names to their metrics
    """
    # Group spans by label
    labels1 = defaultdict(list)
    labels2 = defaultdict(list)
    
    for span in spans1:
        labels1[span.labelname].append(span)
    
    for span in spans2:
        labels2[span.labelname].append(span)
    
    # Get all unique labels
    all_labels = set(labels1.keys()) | set(labels2.keys())
    
    # Calculate metrics for each label
    results = {}
    for label in sorted(all_labels):
        l1_spans = labels1.get(label, [])
        l2_spans = labels2.get(label, [])
        results[label] = calculate_iaa_metrics(l1_spans, l2_spans, match_type)
    
    return results


def calculate_attribute_iaa(spans1: List[Span], spans2: List[Span], 
                           match_type: str = 'context') -> Tuple[Dict, Dict, Dict, Dict]:
    """
    Calculate attribute-level IAA for matched spans (Level 2).
    
    For spans that match at the context level (Level 1), evaluate whether
    the annotators assigned the same attribute values.
    
    Args:
        spans1: Spans from annotator 1
        spans2: Spans from annotator 2
        match_type: 'context' or 'context_overlap'
        
    Returns:
        Tuple of (overall_metrics, per_label_metrics, overall_attribute_metrics, parent_attribute_metrics)
    """
    # Find matched pairs
    matched_pairs = []
    matched_indices_2 = set()
    
    if match_type == 'context':
        for s1 in spans1:
            for j, s2 in enumerate(spans2):
                if j not in matched_indices_2 and s1.context_match(s2):
                    matched_pairs.append((s1, s2))
                    matched_indices_2.add(j)
                    break
    elif match_type == 'context_overlap':
        # For overlap, we need to find best matches
        matched_indices_1 = set()
        matched_indices_2 = set()
        
        for i, s1 in enumerate(spans1):
            if i in matched_indices_1:
                continue
            for j, s2 in enumerate(spans2):
                if j not in matched_indices_2 and s1.context_overlap(s2):
                    matched_pairs.append((s1, s2))
                    matched_indices_1.add(i)
                    matched_indices_2.add(j)
                    break
    
    if not matched_pairs:
        return {
            'total_matched_spans': 0,
            'total_attributes': 0,
            'matching_attributes': 0,
            'precision': 0,
            'recall': 0,
            'f1': 0
        }, {}, {}, {}
    
    # Calculate attribute agreement
    total_attributes = 0
    matching_attributes = 0
    per_label_stats = defaultdict(lambda: {
        'total': 0, 
        'matching': 0, 
        'spans': 0,
        'attributes': defaultdict(lambda: {'total': 0, 'matching': 0})
    })
    
    # Track overall per-attribute stats (across all labels)
    overall_attribute_stats = defaultdict(lambda: {'total': 0, 'matching': 0})
    
    # Track per-category attribute stats (legislation, decision, secondary sources)
    category_attribute_stats = defaultdict(lambda: defaultdict(lambda: {'total': 0, 'matching': 0}))
    
    for s1, s2 in matched_pairs:
        # Determine the category: check labelname first, then parent attribute
        labelname_lower = s1.labelname.lower()
        
        if labelname_lower in ['legislation', 'decision', 'secondary sources']:
            category = labelname_lower
        else:
            # Look at parent attribute
            parent_attr = s1.attributes.get('parent', '')
            if parent_attr:
                # Extract top-level parent (first element if comma-separated)
                top_parent = parent_attr.split(',')[0].strip().lower()
                if top_parent in ['legislation', 'decision', 'secondary sources']:
                    category = top_parent
                else:
                    category = 'other'
            else:
                category = 'other'
        
        # Get attributes excluding style parent, labelname
        attrs1 = {k: v for k, v in s1.attributes.items() 
                 if k not in ['style', 'parent', 'labelname']}
        attrs2 = {k: v for k, v in s2.attributes.items() 
                 if k not in ['style', 'parent', 'labelname']}
        
        # Get all unique attribute keys
        all_keys = set(attrs1.keys()) | set(attrs2.keys())
        
        for key in all_keys:
            total_attributes += 1
            per_label_stats[s1.labelname]['total'] += 1
            per_label_stats[s1.labelname]['attributes'][key]['total'] += 1
            overall_attribute_stats[key]['total'] += 1
            category_attribute_stats[category][key]['total'] += 1
            
            val1 = attrs1.get(key, None)
            val2 = attrs2.get(key, None)
            
            if val1 == val2:
                matching_attributes += 1
                per_label_stats[s1.labelname]['matching'] += 1
                per_label_stats[s1.labelname]['attributes'][key]['matching'] += 1
                overall_attribute_stats[key]['matching'] += 1
                category_attribute_stats[category][key]['matching'] += 1
        
        per_label_stats[s1.labelname]['spans'] += 1
    
    # Calculate overall metrics
    agreement = matching_attributes / total_attributes if total_attributes > 0 else 0
    
    overall = {
        'total_matched_spans': len(matched_pairs),
        'total_attributes': total_attributes,
        'matching_attributes': matching_attributes,
        'agreement': agreement,
        'precision': agreement,
        'recall': agreement,
        'f1': agreement
    }
    
    # Calculate per-label metrics
    per_label = {}
    for label, stats in per_label_stats.items():
        agreement_rate = stats['matching'] / stats['total'] if stats['total'] > 0 else 0
        
        # Calculate per-attribute agreement rates
        attributes_detail = {}
        for attr_name, attr_stats in stats['attributes'].items():
            attr_agreement = attr_stats['matching'] / attr_stats['total'] if attr_stats['total'] > 0 else 0
            attributes_detail[attr_name] = {
                'total': attr_stats['total'],
                'matching': attr_stats['matching'],
                'agreement': attr_agreement
            }
        
        per_label[label] = {
            'matched_spans': stats['spans'],
            'total_attributes': stats['total'],
            'matching_attributes': stats['matching'],
            'agreement': agreement_rate,
            'f1': agreement_rate,
            'attributes': attributes_detail
        }
    
    # Calculate overall per-attribute metrics (across all labels)
    overall_attributes = {}
    for attr_name, attr_stats in overall_attribute_stats.items():
        attr_agreement = attr_stats['matching'] / attr_stats['total'] if attr_stats['total'] > 0 else 0
        overall_attributes[attr_name] = {
            'total': attr_stats['total'],
            'matching': attr_stats['matching'],
            'agreement': attr_agreement
        }
    
    # Calculate per-category per-attribute metrics
    category_attributes = {}
    for category_name, attrs in category_attribute_stats.items():
        category_attributes[category_name] = {}
        for attr_name, attr_stats in attrs.items():
            attr_agreement = attr_stats['matching'] / attr_stats['total'] if attr_stats['total'] > 0 else 0
            category_attributes[category_name][attr_name] = {
                'total': attr_stats['total'],
                'matching': attr_stats['matching'],
                'agreement': attr_agreement
            }
    
    return overall, per_label, overall_attributes, category_attributes


def print_attribute_iaa_results(overall: Dict, per_label: Dict[str, Dict], 
                               overall_attributes: Dict[str, Dict],
                               category_attributes: Dict[str, Dict[str, Dict]],
                               match_type: str, file1: str, file2: str):
    """
    Print Level 2 attribute IAA results in a beautiful, readable format.
    """
    # Header
    print("\n" + "╔" + "═" * 78 + "╗")
    print("║" + " " * 12 + "LEVEL 2: ATTRIBUTE-LEVEL INTER-ANNOTATOR AGREEMENT" + " " * 16 + "║")
    print("╚" + "═" * 78 + "╝")
    
    # Files being compared
    print(f"\n📄 Document 1: {os.path.basename(file1)}")
    print(f"📄 Document 2: {os.path.basename(file2)}")
    
    # Method description
    print(f"\n🔍 Analysis Level: ATTRIBUTE AGREEMENT")
    print(f"   → Evaluating attribute values for matched spans (Level 1 matches)")
    print(f"   → Excluded attributes: style, parent, labelname")
    print(f"   → Included: docid, uri, titletype, fragmentid, etc.")
    
    # Overall results
    print("\n" + "─" * 80)
    print("📊 OVERALL ATTRIBUTE AGREEMENT")
    print("─" * 80)
    
    print(f"\n  Matched Spans (from Level 1):  {overall['total_matched_spans']:>4}")
    print(f"  Total Attributes Compared:     {overall['total_attributes']:>4}")
    print(f"  Matching Attribute Values:     {overall['matching_attributes']:>4}")
    
    print(f"\n  {'Metric':<20} {'Value':>10}   {'Percentage':>10}")
    print(f"  {'-'*20} {'-'*10}   {'-'*10}")
    print(f"  {'Agreement Rate':<20} {overall['agreement']:>10.4f}   {overall['agreement']*100:>9.2f}%")
    print(f"  {'F1 Score':<20} {overall['f1']:>10.4f}   {overall['f1']*100:>9.2f}%")
    
    # Category-based attribute breakdown table
    if overall_attributes:
        print("\n" + "─" * 120)
        print("🌐 ATTRIBUTE AGREEMENT BY CATEGORY")
        print("─" * 120)
        
        # Table header
        print(f"\n{'Attribute':<20} {'Legislation':<22} {'Decision':<22} {'Secondary Sources':<22} {'Total':>8} {'Match':>8} {'Agreement':>10}")
        print("─" * 120)
        
        for attr_name in sorted(overall_attributes.keys()):
            overall_data = overall_attributes[attr_name]
            
            # Get data for each category
            leg_data = category_attributes.get('legislation', {}).get(attr_name, None)
            dec_data = category_attributes.get('decision', {}).get(attr_name, None)
            sec_data = category_attributes.get('secondary sources', {}).get(attr_name, None)
            
            # Format category cells as "match/total (XX%)"
            leg_str = f"{leg_data['matching']}/{leg_data['total']} ({leg_data['agreement']*100:.1f}%)" if leg_data else "-"
            dec_str = f"{dec_data['matching']}/{dec_data['total']} ({dec_data['agreement']*100:.1f}%)" if dec_data else "-"
            sec_str = f"{sec_data['matching']}/{sec_data['total']} ({sec_data['agreement']*100:.1f}%)" if sec_data else "-"
            
            print(f"{attr_name:<20} {leg_str:<22} {dec_str:<22} {sec_str:<22} "
                  f"{overall_data['total']:>8} {overall_data['matching']:>8} {overall_data['agreement']*100:>9.2f}%")
    
    print("\n" + "═" * 120 + "\n")


def print_iaa_results(overall: Dict, per_label: Dict[str, Dict], 
                     match_type: str, file1: str, file2: str, context_chars: int):
    """
    Print Level 1 span IAA results in a beautiful, readable format.
    """
    # Header
    print("\n" + "╔" + "═" * 78 + "╗")
    print("║" + " " * 10 + "LEVEL 1: SPAN-LEVEL INTER-ANNOTATOR AGREEMENT (IAA)" + " " * 17 + "║")
    print("╚" + "═" * 78 + "╝")
    
    # Files being compared
    print(f"\n📄 Document 1: {os.path.basename(file1)}")
    print(f"📄 Document 2: {os.path.basename(file2)}")
    
    # Method description
    print(f"\n🔍 Matching Strategy: {match_type.upper()}")
    if match_type == 'context':
        print(f"   → Exact context matching ({context_chars} chars before/after each label)")
        print(f"   → Ensures labels are at the SAME document location")
    elif match_type == 'context_overlap':
        print(f"   → Similarity-based context matching (70% threshold)")
        print(f"   → Allows slight boundary differences between annotators")
    
    # Overall results
    print("\n" + "─" * 80)
    print("📊 OVERALL AGREEMENT")
    print("─" * 80)
    
    if match_type == 'context':
        print(f"\n  Total Labels (Annotator 1):  {overall['annotator1_count']:>4}")
        print(f"  Total Labels (Annotator 2):  {overall['annotator2_count']:>4}")
        print(f"  Matched Labels:              {overall['matched']:>4}")
    else:
        print(f"\n  Total Labels (Annotator 1):  {overall['annotator1_count']:>4}")
        print(f"  Total Labels (Annotator 2):  {overall['annotator2_count']:>4}")
        print(f"  Matched (Annotator 1):       {overall['matched_annotator1']:>4}")
        print(f"  Matched (Annotator 2):       {overall['matched_annotator2']:>4}")
    
    print(f"\n  {'Metric':<15} {'Value':>10}   {'Percentage':>10}")
    print(f"  {'-'*15} {'-'*10}   {'-'*10}")
    print(f"  {'Precision':<15} {overall['precision']:>10.4f}   {overall['precision']*100:>9.2f}%")
    print(f"  {'Recall':<15} {overall['recall']:>10.4f}   {overall['recall']*100:>9.2f}%")
    print(f"  {'F1 Score':<15} {overall['f1']:>10.4f}   {overall['f1']*100:>9.2f}%")
    
    # Per-label results
    print("\n" + "─" * 80)
    print("📋 PER-LABEL BREAKDOWN")
    print("─" * 80)
    
    print(f"\n{'Label':<30} {'A1':>6} {'A2':>6} {'Match':>8} {'F1 Score':>10}")
    print("─" * 80)
    
    for label in sorted(per_label.keys()):
        metrics = per_label[label]
        if match_type == 'context':
            match_str = f"{metrics['matched']}"
        else:
            match_str = f"{metrics['matched_annotator1']}/{metrics['matched_annotator2']}"
        
        # Truncate long label names
        display_label = label if len(label) <= 30 else label[:27] + "..."
        
        print(f"{display_label:<30} {metrics['annotator1_count']:>6} {metrics['annotator2_count']:>6} "
              f"{match_str:>8} {metrics['f1']*100:>9.2f}%")
    
    print("\n" + "═" * 80 + "\n")


def evaluate_iaa(file1: str, file2: str, match_type: str = 'context', context_chars: int = 200):
    """
    Evaluate Inter-Annotator Agreement using context-aware matching.
    
    Args:
        file1: Path to first annotated HTML file
        file2: Path to second annotated HTML file
        match_type: 'context' (strict, RECOMMENDED) or 'context_overlap' (lenient)
        context_chars: Characters before/after label for context (default: 200)
    
    Returns:
        Tuple of (overall_metrics, per_label_metrics)
    """
    print(f"\n🔄 Extracting spans from {os.path.basename(file1)}...")
    spans1 = extract_spans_from_html(file1, context_chars)
    print(f"   ✓ Found {len(spans1)} labeled spans")
    
    print(f"\n🔄 Extracting spans from {os.path.basename(file2)}...")
    spans2 = extract_spans_from_html(file2, context_chars)
    print(f"   ✓ Found {len(spans2)} labeled spans")
    
    # Calculate metrics
    print(f"\n⚙️  Computing {match_type} agreement metrics...")
    overall = calculate_iaa_metrics(spans1, spans2, match_type)
    per_label = calculate_per_label_iaa(spans1, spans2, match_type)
    
    # Print results
    print_iaa_results(overall, per_label, match_type, file1, file2, context_chars)
    
    return overall, per_label

### Main

In [57]:
# Configuration
anno1 = "EG"  # Options: "GL", "EG", "VP"
anno2 = "VP"  # Options: "GL", "EG", "VP"
file_name = f"2019SCC65_annotated" #"2016QCCS1184_annotated" #"1999CanLII7320_annotated" #"2001BCSC1342_annotated" 2019SCC65_annotated
file1 = rf"C:\Users\zakga\OneDrive\Documents\code\labelstudio\annotation\data\Documents_Annotés\{anno1}\{file_name}_{anno1}_v1.html"
file2 = rf"C:\Users\zakga\OneDrive\Documents\code\labelstudio\annotation\data\Documents_Annotés\{anno2}\{file_name}_{anno2}_v1.html"

# Choose evaluation level
# "level1" = Span matching (context-based)
# "level2" = Attribute matching (for matched spans)
# "both" = Both levels
evaluation_level = "both"  # Options: "level1", "level2", "both"

# Choose matching type for Level 1
# "context" = Strict (exact context match, RECOMMENDED)
# "context_overlap" = Lenient (70% context similarity)
match_type = "context"  # Options: "context", "context_overlap"

# Extract spans
print("\n" + "═" * 80)
print("🔄 EXTRACTING ANNOTATIONS...")
print("═" * 80)
spans1 = extract_spans_from_html(file1, context_chars=200)
spans2 = extract_spans_from_html(file2, context_chars=200)
print(f"\n✓ Annotator 1 ({anno1}): {len(spans1)} spans")
print(f"✓ Annotator 2 ({anno2}): {len(spans2)} spans")

# LEVEL 1: SPAN MATCHING
if evaluation_level in ["level1", "both"]:
    print("\n\n" + "🎯 " + "=" * 76)
    print(f"LEVEL 1: SPAN MATCHING ({match_type.upper()})")
    print("=" * 78)
    
    overall_l1 = calculate_iaa_metrics(spans1, spans2, match_type)
    per_label_l1 = calculate_per_label_iaa(spans1, spans2, match_type)
    
    print_iaa_results(overall_l1, per_label_l1, match_type, file1, file2, 200)

# LEVEL 2: ATTRIBUTE MATCHING
if evaluation_level in ["level2", "both"]:
    print("\n\n" + "🎯 " + "=" * 76)
    print(f"LEVEL 2: ATTRIBUTE MATCHING (for Level 1 matches)")
    print("=" * 78)
    
    overall_l2, per_label_l2, overall_attrs_l2, category_attrs_l2 = calculate_attribute_iaa(spans1, spans2, match_type)
    
    print_attribute_iaa_results(overall_l2, per_label_l2, overall_attrs_l2, category_attrs_l2, match_type, file1, file2)

# Summary
if evaluation_level == "both":
    print("\n" + "╔" + "═" * 78 + "╗")
    print("║" + " " * 30 + "SUMMARY" + " " * 41 + "║")
    print("╚" + "═" * 78 + "╝")
    print(f"\n  Level 1 (Span Matching):       F1 = {overall_l1['f1']*100:>6.2f}%")
    print(f"  Level 2 (Attribute Agreement): F1 = {overall_l2['f1']*100:>6.2f}%")
    print("\n" + "═" * 80 + "\n")


════════════════════════════════════════════════════════════════════════════════
🔄 EXTRACTING ANNOTATIONS...
════════════════════════════════════════════════════════════════════════════════

✓ Annotator 1 (EG): 1568 spans
✓ Annotator 2 (VP): 1569 spans


🎯 ============================================================================
LEVEL 1: SPAN MATCHING (CONTEXT)

╔══════════════════════════════════════════════════════════════════════════════╗
║          LEVEL 1: SPAN-LEVEL INTER-ANNOTATOR AGREEMENT (IAA)                 ║
╚══════════════════════════════════════════════════════════════════════════════╝

📄 Document 1: 2019SCC65_annotated_EG_v1.html
📄 Document 2: 2019SCC65_annotated_VP_v1.html

🔍 Matching Strategy: CONTEXT
   → Exact context matching (200 chars before/after each label)
   → Ensures labels are at the SAME document location

────────────────────────────────────────────────────────────────────────────────
📊 OVERALL AGREEMENT
───────────────────────────────────────────────

#### Level 1

In [15]:
def build_match_maps(spans1, spans2):
    """
    Returns:
      exact_pairs: list of (i, j)
      lenient_pairs: list of (i, j) EXCLUDING exact matches
    """
    exact_pairs = []
    used_j_exact = set()

    # Exact matches
    for i, s1 in enumerate(spans1):
        for j, s2 in enumerate(spans2):
            if j not in used_j_exact and s1.context_match(s2):
                exact_pairs.append((i, j))
                used_j_exact.add(j)
                break

    exact_i = {i for i, _ in exact_pairs}
    exact_j = {j for _, j in exact_pairs}

    # Lenient-only matches
    lenient_pairs = []
    for i, s1 in enumerate(spans1):
        if i in exact_i:
            continue
        for j, s2 in enumerate(spans2):
            if j in exact_j:
                continue
            if s1.context_overlap(s2):
                lenient_pairs.append((i, j))

    return exact_pairs, lenient_pairs


In [26]:
def build_side_by_side_view(spans1, spans2):
    """
    Returns rows of:
    (span1_text | span2_text | match_type)
    """
    exact_pairs, lenient_pairs = build_match_maps(spans1, spans2)

    exact_i = {i for i, _ in exact_pairs}
    exact_j = {j for _, j in exact_pairs}

    # Map lenient matches (one-to-one, greedy, order-preserving)
    lenient_map_1 = {}
    lenient_map_2 = {}

    for i, j in lenient_pairs:
        if i not in lenient_map_1 and j not in lenient_map_2:
            lenient_map_1[i] = j
            lenient_map_2[j] = i

    rows = []
    used_j = set()

    # Walk annotator 1 in order
    for i, s1 in enumerate(spans1):
        if i in exact_i:
            continue  # skip exact matches

        if i in lenient_map_1:
            j = lenient_map_1[i]
            rows.append((s1.text, spans2[j].text, "≈ lenient"))
            used_j.add(j)
        else:
            rows.append((s1.text, "", "✗ no match"))

    # Remaining annotator 2 spans
    for j, s2 in enumerate(spans2):
        if j in exact_j or j in used_j:
            continue
        rows.append(("", s2.text, "✗ no match"))

    return rows


In [41]:
import textwrap

def print_side_by_side(rows, width=60):
    sep = " | "

    header = f"{'Annotator 1':<{width}}{sep}{'Annotator 2':<{width}}{sep}Match"
    print(header)
    print("=" * len(header))

    for a1, a2, tag in rows:
        # Wrap text into multiple lines
        a1_lines = textwrap.wrap(a1, width=width) if a1 else [""]
        a2_lines = textwrap.wrap(a2, width=width) if a2 else [""]

        max_lines = max(len(a1_lines), len(a2_lines))

        # Pad shorter one
        a1_lines += [""] * (max_lines - len(a1_lines))
        a2_lines += [""] * (max_lines - len(a2_lines))

        # Print aligned lines
        for l1, l2 in zip(a1_lines, a2_lines):
            print(f"{l1:<{width}}{sep}{l2:<{width}}{sep}{tag}")

        # Separator between span pairs
        print("-" * len(header))


import textwrap

def get_side_by_side(rows, width=60):
    sep = " | "
    header = f"{'Annotator 1':<{width}}{sep}{'Annotator 2':<{width}}{sep}Match"
    output = [header, "=" * len(header)]

    for a1, a2, tag in rows:
        a1_lines = textwrap.wrap(a1, width=width) if a1 else [""]
        a2_lines = textwrap.wrap(a2, width=width) if a2 else [""]

        max_lines = max(len(a1_lines), len(a2_lines))
        a1_lines += [""] * (max_lines - len(a1_lines))
        a2_lines += [""] * (max_lines - len(a2_lines))

        for l1, l2 in zip(a1_lines, a2_lines):
            output.append(f"{l1:<{width}}{sep}{l2:<{width}}{sep}{tag}")

        output.append("-" * len(header))

    return "\n".join(output)




In [55]:
label = input("Label to inspect: ").strip()

# Filter spans by label
s1 = [s for s in spans1 if s.labelname == label]
s2 = [s for s in spans2 if s.labelname == label]

rows = build_side_by_side_view(s1, s2)
print_side_by_side(rows)

Annotator 1                                                  | Annotator 2                                                  | Match
Brouwer, Andrew                                              | Brouwer, Andrew.                                             | ≈ lenient
-----------------------------------------------------------------------------------------------------------------------------------
Coady, Jonathan M                                            | Coady, Jonathan M.                                           | ≈ lenient
-----------------------------------------------------------------------------------------------------------------------------------
Cromwell, Thomas A                                           | Cromwell, Thomas A.                                          | ≈ lenient
-----------------------------------------------------------------------------------------------------------------------------------
DeMarco, Jerry V                                             | D

##### File output in Html

In [60]:
import html
import textwrap
import datetime
from typing import List, Dict, Any, Tuple


def generate_label_match_html(
    spans1: List[Any],
    spans2: List[Any],
    labels: List[str],
    output_file: str = "label_matches_report.html"
) -> None:
    """
    Generate an HTML report with navigation menu and tables for label matches.
    
    Args:
        spans1: First list of spans from annotator 1
        spans2: Second list of spans from annotator 2
        labels: List of label names to analyze
        output_file: Path to save the HTML file
    """
    
    # Get side-by-side data for each label
    label_data = {}
    for label in labels:
        # Filter spans by label
        s1 = [s for s in spans1 if hasattr(s, 'labelname') and s.labelname == label]
        s2 = [s for s in spans2 if hasattr(s, 'labelname') and s.labelname == label]
        
        # Build rows for side-by-side view
        rows = build_side_by_side_view(s1, s2)
        label_data[label] = {
            'rows': rows,
            'count_a1': len(s1),
            'count_a2': len(s2),
            'matches': sum(1 for _, _, tag in rows if tag == "✓"),
            'mismatches': sum(1 for _, _, tag in rows if tag == "✗"),
            'only_a1': sum(1 for _, _, tag in rows if tag == "Only in A1"),
            'only_a2': sum(1 for _, _, tag in rows if tag == "Only in A2")
        }
    
    # Get current timestamp
    current_time = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    # Generate HTML
    html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Label Matches Report</title>
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }}
        
        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            line-height: 1.6;
            color: #333;
            background-color: #f5f7fa;
            padding: 20px;
        }}
        
        .container {{
            max-width: 1600px;
            margin: 0 auto;
        }}
        
        .header {{
            text-align: center;
            margin-bottom: 30px;
            padding: 20px;
            background: linear-gradient(135deg, #2c3e50 0%, #3498db 100%);
            color: white;
            border-radius: 10px;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
        }}
        
        .header h1 {{
            font-size: 2.5em;
            margin-bottom: 10px;
        }}
        
        .header .stats {{
            font-size: 1.1em;
            opacity: 0.9;
        }}
        
        .summary {{
            display: flex;
            flex-wrap: wrap;
            gap: 15px;
            justify-content: center;
            margin: 20px 0;
        }}
        
        .summary-item {{
            background: white;
            padding: 15px 25px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0, 0, 0, 0.1);
            text-align: center;
            min-width: 120px;
        }}
        
        .summary-value {{
            font-size: 1.8em;
            font-weight: bold;
            margin-bottom: 5px;
        }}
        
        .summary-label {{
            font-size: 0.9em;
            color: #7f8c8d;
        }}
        
        .nav-menu {{
            position: sticky;
            top: 20px;
            background: white;
            padding: 20px;
            border-radius: 10px;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
            margin-bottom: 30px;
            z-index: 1000;
        }}
        
        .nav-menu h2 {{
            color: #2c3e50;
            margin-bottom: 15px;
            padding-bottom: 10px;
            border-bottom: 2px solid #3498db;
        }}
        
        .nav-links {{
            display: flex;
            flex-wrap: wrap;
            gap: 15px;
        }}
        
        .nav-link {{
            padding: 10px 20px;
            background: #3498db;
            color: white;
            text-decoration: none;
            border-radius: 5px;
            font-weight: 500;
            transition: all 0.3s ease;
            white-space: nowrap;
            display: flex;
            flex-direction: column;
            align-items: center;
        }}
        
        .nav-link:hover {{
            background: #2980b9;
            transform: translateY(-2px);
            box-shadow: 0 4px 8px rgba(0, 0, 0, 0.2);
        }}
        
        .nav-link .label-name {{
            font-size: 1.1em;
            margin-bottom: 5px;
        }}
        
        .nav-link .label-stats {{
            font-size: 0.85em;
            opacity: 0.9;
        }}
        
        .section {{
            margin-bottom: 40px;
            background: white;
            border-radius: 10px;
            padding: 25px;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
        }}
        
        .section-header {{
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 20px;
            padding-bottom: 15px;
            border-bottom: 2px solid #ecf0f1;
            flex-wrap: wrap;
            gap: 15px;
        }}
        
        .section-title {{
            font-size: 1.8em;
            color: #2c3e50;
        }}
        
        .label-stats-container {{
            display: flex;
            gap: 15px;
            flex-wrap: wrap;
        }}
        
        .stat-badge {{
            padding: 5px 15px;
            border-radius: 20px;
            font-weight: bold;
            font-size: 0.9em;
        }}
        
        .match-stat {{
            background: #27ae60;
            color: white;
        }}
        
        .mismatch-stat {{
            background: #e74c3c;
            color: white;
        }}
        
        .only-a1-stat {{
            background: #3498db;
            color: white;
        }}
        
        .only-a2-stat {{
            background: #9b59b6;
            color: white;
        }}
        
        .back-to-top {{
            display: inline-block;
            margin-top: 20px;
            padding: 8px 16px;
            background: #2c3e50;
            color: white;
            text-decoration: none;
            border-radius: 5px;
            font-size: 0.9em;
            transition: background 0.3s ease;
        }}
        
        .back-to-top:hover {{
            background: #1a252f;
        }}
        
        table {{
            width: 100%;
            border-collapse: collapse;
            margin-top: 15px;
        }}
        
        th {{
            background: #34495e;
            color: white;
            padding: 15px;
            text-align: left;
            font-weight: 600;
        }}
        
        td {{
            padding: 15px;
            border-bottom: 1px solid #ecf0f1;
            vertical-align: top;
        }}
        
        tr:hover {{
            background-color: #f8f9fa;
        }}
        
        .annotator-text {{
            max-width: 600px;
            word-wrap: break-word;
        }}
        
        .a1-text {{
            background: #e8f4fc;
            padding: 10px;
            border-radius: 4px;
            border-left: 3px solid #3498db;
        }}
        
        .a2-text {{
            background: #f4e8fc;
            padding: 10px;
            border-radius: 4px;
            border-left: 3px solid #9b59b6;
        }}
        
        .match-cell {{
            text-align: center;
            font-weight: bold;
            font-size: 1.2em;
            width: 80px;
        }}
        
        .match-check {{
            color: #27ae60;
        }}
        
        .match-cross {{
            color: #e74c3c;
        }}
        
        .match-only {{
            color: #7f8c8d;
            font-size: 0.9em;
            font-style: italic;
        }}
        
        .empty-cell {{
            color: #bdc3c7;
            font-style: italic;
            background: #f8f9fa;
        }}
        
        .footer {{
            text-align: center;
            margin-top: 50px;
            padding: 20px;
            color: #7f8c8d;
            font-size: 0.9em;
            border-top: 1px solid #ecf0f1;
        }}
        
        @media (max-width: 768px) {{
            .nav-links {{
                flex-direction: column;
            }}
            
            .nav-link {{
                width: 100%;
                text-align: center;
            }}
            
            table {{
                display: block;
                overflow-x: auto;
            }}
            
            th, td {{
                padding: 10px;
                font-size: 0.9em;
            }}
            
            .annotator-text {{
                max-width: 250px;
            }}
        }}
    </style>
</head>
<body>
    <div class="container">
        <header class="header">
            <h1>Label Matches Report</h1>
            <div class="stats">
                Comparing {len(labels)} labels between Annotator 1 and Annotator 2
            </div>
        </header>
        
        <div class="summary">
            <div class="summary-item">
                <div class="summary-value">{len(labels)}</div>
                <div class="summary-label">Labels</div>
            </div>
            <div class="summary-item">
                <div class="summary-value">{sum(data['count_a1'] for data in label_data.values())}</div>
                <div class="summary-label">Total A1 Spans</div>
            </div>
            <div class="summary-item">
                <div class="summary-value">{sum(data['count_a2'] for data in label_data.values())}</div>
                <div class="summary-label">Total A2 Spans</div>
            </div>
            <div class="summary-item">
                <div class="summary-value">{sum(data['matches'] for data in label_data.values())}</div>
                <div class="summary-label">Perfect Matches</div>
            </div>
            <div class="summary-item">
                <div class="summary-value">{sum(data['mismatches'] for data in label_data.values())}</div>
                <div class="summary-label">Mismatches</div>
            </div>
        </div>
        
        <nav class="nav-menu" id="nav-menu">
            <h2>Quick Navigation</h2>
            <div class="nav-links">
"""
    
    # Add navigation links for each label
    for label in labels:
        data = label_data[label]
        html_content += f"""                <a href="#{label}" class="nav-link">
                    <span class="label-name">{label.title()}</span>
                    <span class="label-stats">{data['count_a1']} vs {data['count_a2']} spans</span>
                </a>
"""
    
    html_content += """            </div>
        </nav>
        
        <main>
"""
    
    # Add sections for each label
    for label in labels:
        data = label_data[label]
        rows = data['rows']
        
        html_content += f"""            <section class="section" id="{label}">
                <div class="section-header">
                    <h2 class="section-title">{label.title()}</h2>

                </div>
"""
        
        if rows:
            html_content += """                <table>
                    <thead>
                        <tr>
                            <th style="width: 45%">Annotator 1</th>
                            <th style="width: 45%">Annotator 2</th>
                            <th style="width: 10%">Match</th>
                        </tr>
                    </thead>
                    <tbody>
"""
            
            for a1_text, a2_text, tag in rows:
                # Escape HTML and wrap text
                escaped_a1 = html.escape(a1_text) if a1_text else ""
                escaped_a2 = html.escape(a2_text) if a2_text else ""
                
                # Determine CSS class for match status
                if tag == "✓":
                    match_class = "match-check"
                    match_display = "✓"
                elif tag == "✗":
                    match_class = "match-cross"
                    match_display = "✗"
                elif tag == "Only in A1":
                    match_class = "match-only"
                    match_display = "Only A1"
                elif tag == "Only in A2":
                    match_class = "match-only"
                    match_display = "Only A2"
                else:
                    match_class = ""
                    match_display = tag
                
                html_content += f"""                        <tr>
                            <td>
                                <div class="annotator-text">
                                    <div class="{'a1-text' if escaped_a1 else 'empty-cell'}">
                                        {escaped_a1 if escaped_a1 else '∅'}
                                    </div>
                                </div>
                            </td>
                            <td>
                                <div class="annotator-text">
                                    <div class="{'a2-text' if escaped_a2 else 'empty-cell'}">
                                        {escaped_a2 if escaped_a2 else '∅'}
                                    </div>
                                </div>
                            </td>
                            <td class="match-cell {match_class}">{match_display}</td>
                        </tr>
"""
            
            html_content += """                    </tbody>
                </table>
"""
        else:
            html_content += """                <div style="text-align: center; padding: 40px; color: #7f8c8d;">
                    <h3>No spans found for this label</h3>
                    <p>Neither annotator has any spans with this label.</p>
                </div>
"""
        
        html_content += f"""                <a href="#nav-menu" class="back-to-top">Back to Navigation Menu ↑</a>
            </section>
"""
    
    # Add footer
    html_content += f"""        </main>
        
        <footer class="footer">
            <p>Report generated on {current_time}</p>
            <p>Total labels analyzed: {len(labels)} | Annotator 1 total spans: {sum(data['count_a1'] for data in label_data.values())} | Annotator 2 total spans: {sum(data['count_a2'] for data in label_data.values())}</p>
        </footer>
    </div>
    
    <script>
        // Make navigation menu sticky
        window.addEventListener('scroll', function() {{
            const navMenu = document.getElementById('nav-menu');
            const scrollTop = window.pageYOffset || document.documentElement.scrollTop;
            
            if (scrollTop > 100) {{
                navMenu.style.top = '0px';
                navMenu.style.boxShadow = '0 2px 10px rgba(0, 0, 0, 0.1)';
            }} else {{
                navMenu.style.top = '20px';
                navMenu.style.boxShadow = '0 4px 6px rgba(0, 0, 0, 0.1)';
            }}
        }});
        
        // Smooth scrolling for anchor links
        document.querySelectorAll('a[href^="#"]').forEach(anchor => {{
            anchor.addEventListener('click', function (e) {{
                e.preventDefault();
                const targetId = this.getAttribute('href');
                if (targetId === '#') return;
                
                const targetElement = document.querySelector(targetId);
                if (targetElement) {{
                    window.scrollTo({{
                        top: targetElement.offsetTop - 20,
                        behavior: 'smooth'
                    }});
                }}
            }});
        }});
        
        // Highlight rows on hover
        document.querySelectorAll('tr').forEach(row => {{
            row.addEventListener('mouseenter', function() {{
                this.style.backgroundColor = '#f0f7ff';
            }});
            row.addEventListener('mouseleave', function() {{
                this.style.backgroundColor = '';
            }});
        }});
    </script>
</body>
</html>"""
    
    # Save HTML file
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"HTML report saved to: {output_file}")
    print(f"Open {output_file} in your browser to view the report.")






    
# List of labels to analyze
labels = [
    "decision",
    "legislation",
    "secondary sources",
    "title",
    "fragment",
    "reference",
    "source",
    "authors"
]

# Generate the HTML report
generate_label_match_html(spans1, spans2, labels, "label_matches_report.html")

HTML report saved to: label_matches_report.html
Open label_matches_report.html in your browser to view the report.


#### Level 2

In [49]:
def get_matched_span_pairs(spans1, spans2, match_type="context"):
    matched_pairs = []
    used_2 = set()

    if match_type == "context":
        for s1 in spans1:
            for j, s2 in enumerate(spans2):
                if j not in used_2 and s1.context_match(s2):
                    matched_pairs.append((s1, s2))
                    used_2.add(j)
                    break

    elif match_type == "context_overlap":
        used_1 = set()
        for i, s1 in enumerate(spans1):
            if i in used_1:
                continue
            for j, s2 in enumerate(spans2):
                if j not in used_2 and s1.context_overlap(s2):
                    matched_pairs.append((s1, s2))
                    used_1.add(i)
                    used_2.add(j)
                    break
    else:
        raise ValueError("Unknown match_type")

    return matched_pairs


In [50]:
def get_attribute_mismatches(
    spans1,
    spans2,
    attribute_name,
    match_type="context"
):
    mismatches = []

    matched_pairs = get_matched_span_pairs(spans1, spans2, match_type)

    for s1, s2 in matched_pairs:
        v1 = s1.attributes.get(attribute_name)
        v2 = s2.attributes.get(attribute_name)

        if v1 != v2:
            mismatches.append({
                "span_text": s1.text,
                "a1_value": v1,
                "a2_value": v2,
                "label": s1.labelname
            })

    return mismatches


In [51]:
import textwrap

def print_attribute_side_by_side(mismatches, width=60):
    sep = " | "
    header = (
        f"{'Span text':<{width}}{sep}"
        f"{'Annotator 1':<{width}}{sep}"
        f"{'Annotator 2':<{width}}"
    )
    print(header)
    print("=" * len(header))

    for m in mismatches:
        span_lines = textwrap.wrap(m["span_text"], width) or [""]
        a1_lines = textwrap.wrap(str(m["a1_value"]), width) if m["a1_value"] else ["∅"]
        a2_lines = textwrap.wrap(str(m["a2_value"]), width) if m["a2_value"] else ["∅"]

        max_lines = max(len(span_lines), len(a1_lines), len(a2_lines))

        span_lines += [""] * (max_lines - len(span_lines))
        a1_lines += [""] * (max_lines - len(a1_lines))
        a2_lines += [""] * (max_lines - len(a2_lines))

        for l1, l2, l3 in zip(span_lines, a1_lines, a2_lines):
            print(f"{l1:<{width}}{sep}{l2:<{width}}{sep}{l3:<{width}}")

        print("-" * len(header))


In [53]:
attribute = input("Attribute name to inspect (e.g. docid): ").strip()

mismatches = get_attribute_mismatches(
    spans1,
    spans2,
    attribute_name=attribute,
    match_type="context"  # or "context_overlap"
)

print(f"\nFound {len(mismatches)} mismatches for attribute '{attribute}'\n")
print_attribute_side_by_side(mismatches)



Found 229 mismatches for attribute 'uri'

Span text                                                    | Annotator 1                                                  | Annotator 2                                                 
Canada (Minister of Citizenship and Immigration) v. Vavilov  | ∅                                                            | https://www.canlii.org/en/ca/scc/doc/2019/2019scc65/2019scc6
                                                             |                                                              | 5.html                                                      
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
2019 SCC 65                                                  | ∅                                                            | https://www.canlii.org/en/ca/scc/doc/2019/2019scc65/2019scc6
                      

##### File output in html

In [ ]:
import html
import textwrap
import datetime
from typing import List, Dict, Any

def generate_attribute_mismatch_html(
    spans1: List[Dict[str, Any]],
    spans2: List[Dict[str, Any]],
    attributes: List[str],
    output_file: str = "mismatches_report.html"
) -> None:
    """
    Generate an HTML report with navigation menu and tables for attribute mismatches.
    
    Args:
        spans1: First list of spans
        spans2: Second list of spans
        attributes: List of attribute names to analyze
        output_file: Path to save the HTML file
    """
    
    # Get mismatches for each attribute
    attribute_data = {}
    for attr in attributes:
        mismatches = get_attribute_mismatches(
            spans1,
            spans2,
            attribute_name=attr,
            match_type="context"
        )
        attribute_data[attr] = mismatches
    
    # Get current timestamp
    current_time = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    # Generate HTML
    html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Attribute Mismatches Report</title>
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }}
        
        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            line-height: 1.6;
            color: #333;
            background-color: #f5f7fa;
            padding: 20px;
        }}
        
        .container {{
            max-width: 1400px;
            margin: 0 auto;
        }}
        
        .header {{
            text-align: center;
            margin-bottom: 30px;
            padding: 20px;
            background: linear-gradient(135deg, #6a11cb 0%, #2575fc 100%);
            color: white;
            border-radius: 10px;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
        }}
        
        .header h1 {{
            font-size: 2.5em;
            margin-bottom: 10px;
        }}
        
        .header .stats {{
            font-size: 1.1em;
            opacity: 0.9;
        }}
        
        .nav-menu {{
            position: sticky;
            top: 20px;
            background: white;
            padding: 20px;
            border-radius: 10px;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
            margin-bottom: 30px;
            z-index: 1000;
        }}
        
        .nav-menu h2 {{
            color: #2c3e50;
            margin-bottom: 15px;
            padding-bottom: 10px;
            border-bottom: 2px solid #3498db;
        }}
        
        .nav-links {{
            display: flex;
            flex-wrap: wrap;
            gap: 15px;
        }}
        
        .nav-link {{
            padding: 10px 20px;
            background: #3498db;
            color: white;
            text-decoration: none;
            border-radius: 5px;
            font-weight: 500;
            transition: all 0.3s ease;
            white-space: nowrap;
        }}
        
        .nav-link:hover {{
            background: #2980b9;
            transform: translateY(-2px);
            box-shadow: 0 4px 8px rgba(0, 0, 0, 0.2);
        }}
        
        .section {{
            margin-bottom: 40px;
            background: white;
            border-radius: 10px;
            padding: 25px;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
        }}
        
        .section-header {{
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 20px;
            padding-bottom: 15px;
            border-bottom: 2px solid #ecf0f1;
        }}
        
        .section-title {{
            font-size: 1.8em;
            color: #2c3e50;
        }}
        
        .mismatch-count {{
            background: #e74c3c;
            color: white;
            padding: 5px 15px;
            border-radius: 20px;
            font-weight: bold;
        }}
        
        .back-to-top {{
            display: inline-block;
            margin-top: 20px;
            padding: 8px 16px;
            background: #2c3e50;
            color: white;
            text-decoration: none;
            border-radius: 5px;
            font-size: 0.9em;
            transition: background 0.3s ease;
        }}
        
        .back-to-top:hover {{
            background: #1a252f;
        }}
        
        table {{
            width: 100%;
            border-collapse: collapse;
            margin-top: 15px;
        }}
        
        th {{
            background: #34495e;
            color: white;
            padding: 15px;
            text-align: left;
            font-weight: 600;
        }}
        
        td {{
            padding: 15px;
            border-bottom: 1px solid #ecf0f1;
            vertical-align: top;
        }}
        
        tr:hover {{
            background-color: #f8f9fa;
        }}
        
        .span-text {{
            max-width: 400px;
            word-wrap: break-word;
            background: #f8f9fa;
            padding: 8px;
            border-radius: 4px;
            border-left: 3px solid #3498db;
        }}
        
        .value {{
            max-width: 300px;
            word-wrap: break-word;
        }}
        
        .empty-value {{
            color: #7f8c8d;
            font-style: italic;
        }}
        
        .footer {{
            text-align: center;
            margin-top: 50px;
            padding: 20px;
            color: #7f8c8d;
            font-size: 0.9em;
            border-top: 1px solid #ecf0f1;
        }}
        
        @media (max-width: 768px) {{
            .nav-links {{
                flex-direction: column;
            }}
            
            .nav-link {{
                width: 100%;
                text-align: center;
            }}
            
            table {{
                display: block;
                overflow-x: auto;
            }}
            
            th, td {{
                padding: 10px;
                font-size: 0.9em;
            }}
        }}
    </style>
</head>
<body>
    <div class="container">
        <header class="header">
            <h1>Attribute Mismatches Report</h1>
            <div class="stats">
                Generated for {len(attributes)} attributes
            </div>
        </header>
        
        <nav class="nav-menu" id="nav-menu">
            <h2>Quick Navigation</h2>
            <div class="nav-links">
"""
    
    # Add navigation links
    for attr in attributes:
        mismatches = attribute_data[attr]
        html_content += f'                <a href="#{attr}" class="nav-link">{attr.title()} ({len(mismatches)} mismatches)</a>\n'
    
    html_content += """            </div>
        </nav>
        
        <main>
"""
    
    # Add sections for each attribute
    for attr in attributes:
        mismatches = attribute_data[attr]
        
        html_content += f"""            <section class="section" id="{attr}">
                <div class="section-header">
                    <h2 class="section-title">{attr.title()} Mismatches</h2>
                    <span class="mismatch-count">{len(mismatches)} mismatches</span>
                </div>
"""
        
        if mismatches:
            html_content += """                <table>
                    <thead>
                        <tr>
                            <th style="width: 30%">Span Text</th>
                            <th style="width: 35%">Annotator 1 Value</th>
                            <th style="width: 35%">Annotator 2 Value</th>
                        </tr>
                    </thead>
                    <tbody>
"""
            
            for m in mismatches:
                span_text = html.escape(m.get("span_text", ""))
                a1_value = html.escape(str(m.get("a1_value", ""))) if m.get("a1_value") else "<span class='empty-value'>∅</span>"
                a2_value = html.escape(str(m.get("a2_value", ""))) if m.get("a2_value") else "<span class='empty-value'>∅</span>"
                
                html_content += f"""                        <tr>
                            <td><div class="span-text">{span_text}</div></td>
                            <td><div class="value">{a1_value}</div></td>
                            <td><div class="value">{a2_value}</div></td>
                        </tr>
"""
            
            html_content += """                    </tbody>
                </table>
"""
        else:
            html_content += """                <div style="text-align: center; padding: 40px; color: #27ae60;">
                    <h3>✅ No mismatches found for this attribute</h3>
                    <p>Both annotators agree on all values for this attribute.</p>
                </div>
"""
        
        html_content += f"""                <a href="#nav-menu" class="back-to-top">Back to Navigation Menu ↑</a>
            </section>
"""
    
    # Add footer
    html_content += f"""        </main>
        
        <footer class="footer">
            <p>Report generated on {current_time}</p>
            <p>Total attributes analyzed: {len(attributes)}</p>
        </footer>
    </div>
    
    <script>
        // Make navigation menu sticky
        window.addEventListener('scroll', function() {{
            const navMenu = document.getElementById('nav-menu');
            const scrollTop = window.pageYOffset || document.documentElement.scrollTop;
            
            if (scrollTop > 100) {{
                navMenu.style.top = '0px';
                navMenu.style.boxShadow = '0 2px 10px rgba(0, 0, 0, 0.1)';
            }} else {{
                navMenu.style.top = '20px';
                navMenu.style.boxShadow = '0 4px 6px rgba(0, 0, 0, 0.1)';
            }}
        }});
        
        // Smooth scrolling for anchor links
        document.querySelectorAll('a[href^="#"]').forEach(anchor => {{
            anchor.addEventListener('click', function (e) {{
                e.preventDefault();
                const targetId = this.getAttribute('href');
                if (targetId === '#') return;
                
                const targetElement = document.querySelector(targetId);
                if (targetElement) {{
                    window.scrollTo({{
                        top: targetElement.offsetTop - 20,
                        behavior: 'smooth'
                    }});
                }}
            }});
        }});
    </script>
</body>
</html>"""
    
    # Save HTML file
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"HTML report saved to: {output_file}")
    print(f"Open {output_file} in your browser to view the report.")




    
# List of attributes to analyze
attributes = ["docid", "uri", "fragmentid", "non_standard", "titletype"]

# Generate the HTML report
generate_attribute_mismatch_html(spans1, spans2, attributes, "attributes_mismatches_report.html")

HTML report saved to: mismatches_report.html
Open mismatches_report.html in your browser to view the report.
